# Notebook B — 7B Inference for Missing Methods

Runs 7B inference for methods that need it:
- M2v2: WED, WDE, EWD, EDW
- M3 Juggler v1
- M3v2 Juggler
- Raw baseline (no adapter)

**Updates existing inference files** to add the 7B data alongside 1.5B and 3B.
Uses 8-bit quantization for 7B (same as Phase 1).

In [1]:
# Cell 0: Config + load questions
import os, json, time, gc
import torch

PROJECT_DIR = r"C:\Users\adishalit1\Desktop\kd_project"
# PROJECT_DIR = os.path.expanduser("~/kd_project")  # Linux

DATA_DIR = os.path.join(PROJECT_DIR, "data")
RUNS_DIR = os.path.join(PROJECT_DIR, "runs")
N_EVAL = 100
GEN_KW = dict(max_new_tokens=2000, do_sample=False)

QUESTIONS_FILE = os.path.join(DATA_DIR, "eval_questions_100.json")
with open(QUESTIONS_FILE) as f:
    q_data = json.load(f)
eval_questions = q_data["questions"]

SNAME = "qwen25_7b_base"
SPATH = "Qwen/Qwen2.5-7B"

print(f"GPU: {torch.cuda.get_device_name() if torch.cuda.is_available() else 'NONE'}")
print(f"Will run 7B inference for missing methods")

GPU: NVIDIA GeForce RTX 4090
Will run 7B inference for missing methods


In [2]:
# Cell 1: Define which experiments need 7B
# Path resolution: maps exp_name -> adapter folder (or None for raw)

EXPERIMENTS_7B = {
    "raw_baseline": None,
    "m3_juggler": os.path.join(RUNS_DIR, "m3_juggler", SNAME),
    "m3v2_juggler": os.path.join(RUNS_DIR, "m3v2_juggler", SNAME),
    "m2v2_WED": os.path.join(RUNS_DIR, "m2v2_sequential", SNAME, "WED"),
    "m2v2_WDE": os.path.join(RUNS_DIR, "m2v2_sequential", SNAME, "WDE"),
    "m2v2_EWD": os.path.join(RUNS_DIR, "m2v2_sequential", SNAME, "EWD"),
    "m2v2_EDW": os.path.join(RUNS_DIR, "m2v2_sequential", SNAME, "EDW"),
    "m1v2_A1": os.path.join(RUNS_DIR, "m1v2_additive_grid", SNAME, "A1"),
}

def find_adapter(base):
    """For M2v2, check final/stage3 subdirs."""
    if base is None:
        return None
    # Direct adapter?
    if os.path.exists(os.path.join(base, "adapter_config.json")):
        return base
    # M2v2: try final/stage3
    for sub in ["final", "stage3"]:
        path = os.path.join(base, sub)
        if os.path.exists(os.path.join(path, "adapter_config.json")):
            return path
    return None

# Check what's actually available and what needs running
to_run = {}
for exp_name, adapter_base in EXPERIMENTS_7B.items():
    if adapter_base is None:
        resolved = None  # raw
    else:
        resolved = find_adapter(adapter_base)
        if resolved is None:
            print(f"  ⚠️ {exp_name}: adapter not found at {adapter_base}")
            continue

    # Check if 7B is already in the inference file
    inf_file = os.path.join(DATA_DIR, f"{exp_name}_inference_{N_EVAL}_TESTONLY.json")
    already_done = False
    if os.path.exists(inf_file):
        with open(inf_file) as f:
            existing = json.load(f)
        if SNAME in existing.get("models", {}):
            already_done = True

    if already_done:
        print(f"  ⏩ {exp_name}: 7B already done")
    else:
        to_run[exp_name] = resolved
        status = "raw model" if resolved is None else f"adapter at {resolved}"
        print(f"  🔄 {exp_name}: will run ({status})")

print(f"\nTotal to run: {len(to_run)}")

  ⏩ raw_baseline: 7B already done
  ⏩ m3_juggler: 7B already done
  ⏩ m3v2_juggler: 7B already done
  ⏩ m2v2_WED: 7B already done
  ⏩ m2v2_WDE: 7B already done
  ⏩ m2v2_EWD: 7B already done
  ⏩ m2v2_EDW: 7B already done
  🔄 m1v2_A1: will run (adapter at C:\Users\adishalit1\Desktop\kd_project\runs\m1v2_additive_grid\qwen25_7b_base\A1)

Total to run: 1


In [ ]:
# Cell 2: Load 7B base once, run all adapters
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import PeftModel

if not to_run:
    print("✅ Nothing to do")
else:
    print(f"\nLoading 7B base model (8-bit)...")
    tokenizer = AutoTokenizer.from_pretrained(SPATH, trust_remote_code=True)
    if tokenizer.pad_token is None: tokenizer.pad_token = tokenizer.eos_token

    bnb = BitsAndBytesConfig(load_in_8bit=True)
    base_model = AutoModelForCausalLM.from_pretrained(
        SPATH, quantization_config=bnb, device_map="auto", trust_remote_code=True)
    base_model.eval()
    print(f"✅ Base loaded")

    @torch.inference_mode()
    def run_inference(model):
        answers = []
        for q in eval_questions:
            inputs = tokenizer(q["prompt"], return_tensors="pt", truncation=True)
            inputs = {k: v.to("cuda") for k, v in inputs.items()}
            out = model.generate(**inputs, **GEN_KW)
            decoded = tokenizer.decode(out[0], skip_special_tokens=True)
            answer = decoded[len(q["prompt"]):].lstrip() if decoded.startswith(q["prompt"]) else decoded.strip()
            answers.append(answer)
        return answers

    for exp_name, adapter_path in to_run.items():
        print(f"\n{'='*60}")
        print(f"  {exp_name}")
        print(f"{'='*60}")

        if adapter_path is None:
            model = base_model
            print("  Raw baseline (no adapter)")
        else:
            print(f"  Loading adapter: {adapter_path}")
            model = PeftModel.from_pretrained(base_model, adapter_path)
            model.eval()

        t0 = time.time()
        answers = run_inference(model)
        elapsed = time.time() - t0
        print(f"  ✅ {len(answers)} samples in {elapsed/60:.1f} min ({elapsed/len(answers):.1f}s each)")

        # Update inference file (or create if missing)
        inf_file = os.path.join(DATA_DIR, f"{exp_name}_inference_{N_EVAL}_TESTONLY.json")
        if os.path.exists(inf_file):
            with open(inf_file) as f:
                result = json.load(f)
        else:
            result = {
                "meta": {"source": "eval_questions_100.json", "n_samples": N_EVAL},
                "models": {},
                "samples": [{"id": q["id"], "prompt": q["prompt"], "outputs": {}} for q in eval_questions],
            }

        result["models"][SNAME] = {"adapter": adapter_path or "raw", "path": SPATH}
        for sample, answer in zip(result["samples"], answers):
            sample["outputs"][SNAME] = {"answer": answer}

        with open(inf_file, "w", encoding="utf-8") as f:
            json.dump(result, f, ensure_ascii=False, indent=2)
        print(f"  Saved → {inf_file}")

        # Cleanup adapter (keep base model)
        if adapter_path is not None:
            del model
            gc.collect()
            torch.cuda.empty_cache()

    del base_model, tokenizer
    gc.collect()
    torch.cuda.empty_cache()
    print("\n✅ All 7B inference done")

c:\Users\adishalit1\AppData\Local\anaconda3\envs\kd\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm



Loading 7B base model (8-bit)...


Loading weights: 100%|██████████| 339/339 [00:15<00:00, 22.14it/s]
Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.


✅ Base loaded

  m1v2_A1
  Loading adapter: C:\Users\adishalit1\Desktop\kd_project\runs\m1v2_additive_grid\qwen25_7b_base\A1


c:\Users\adishalit1\AppData\Local\anaconda3\envs\kd\Lib\site-packages\bitsandbytes\autograd\_functions.py:123: UserWarning: MatMul8bitLt: inputs will be cast from torch.bfloat16 to float16 during quantization
  warnings.warn(f"MatMul8bitLt: inputs will be cast from {A.dtype} to float16 during quantization")
Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.
Setting `pad_token_id` to `eos_tok

  ✅ 100 samples in 47.3 min (28.4s each)
  Saved → C:\Users\adishalit1\Desktop\kd_project\data\m1v2_A1_inference_100_TESTONLY.json

✅ All 7B inference done


: 